# Análisis de Sentimiento y Discurso de Odio — Boric vs Kast

Compara la cobertura de prensa de los primeros días del mandato de Gabriel Boric (2022) y José Antonio Kast (2026).
Mismo período del calendario (11 marzo al 8 mayo).

## 1. Imports

In [ ]:
import glob
import re
import html

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

tqdm.pandas()
pd.set_option('display.max_colwidth', 120)

## 2. Carga del dataset

In [ ]:
# Carga el CSV de gobierno (Boric + Kast)
df = pd.read_csv('../resultados_gobierno.csv', encoding='utf-8')

print(f'Dimensiones del dataset: {df.shape}')
print(f"Períodos: {df['ID recolección'].unique().tolist()}")
df.head(3)

In [ ]:
df.info()

In [ ]:
TEXT_COL = 'Texto'
GROUP_COL = 'ID recolección'   # boric 1 / kast 1

print(f'Valores nulos en "{TEXT_COL}": {df[TEXT_COL].isna().sum()}')
print(f'\nDistribución por período:')
print(df[GROUP_COL].value_counts())

## 3. Limpieza del texto

In [ ]:
def clean_text(raw: str) -> str:
    """Limpia texto con HTML, entidades y ruido tipográfico."""
    if not isinstance(raw, str):
        return ''

    text = html.unescape(raw)
    text = BeautifulSoup(text, 'lxml').get_text(separator=' ')
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'[@#]\w+', '', text)
    text = re.sub(r'[\x00-\x1f\x7f-\x9f]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    return text


df['texto_limpio'] = df[TEXT_COL].progress_apply(clean_text)
df = df[df['texto_limpio'].str.len() > 0].reset_index(drop=True)

print(f'Filas tras limpieza: {len(df)}')
print('\nEjemplo limpio:')
print(df['texto_limpio'].iloc[0][:400])

## 4. Análisis de sentimiento

`pysentimiento` trunca a 512 tokens. Para artículos largos esto solo cubre el inicio del texto.

In [ ]:
from pysentimiento import create_analyzer

sentiment_analyzer = create_analyzer(task='sentiment', lang='es')

In [ ]:
def run_sentiment(text: str) -> dict:
    result = sentiment_analyzer.predict(text)
    return {
        'sent_label': result.output,
        'sent_pos': round(result.probas.get('POS', 0), 4),
        'sent_neu': round(result.probas.get('NEU', 0), 4),
        'sent_neg': round(result.probas.get('NEG', 0), 4),
    }

print('Ejecutando análisis de sentimiento...')
sent_results = df['texto_limpio'].progress_apply(run_sentiment)
df = pd.concat([df, pd.DataFrame(list(sent_results))], axis=1)
print('Listo.')
df[['texto_limpio', 'sent_label', 'sent_pos', 'sent_neu', 'sent_neg']].head(5)

## 5. Detección de discurso de odio

In [ ]:
hate_analyzer = create_analyzer(task='hate_speech', lang='es')
print('Analizador de discurso de odio cargado.')

In [ ]:
def run_hate(text: str) -> dict:
    result = hate_analyzer.predict(text)
    return {
        'hate_label': result.output,
        'hate_score': round(result.probas.get('hateful', result.probas.get('HATEFUL', 0)), 4),
    }

print('Ejecutando detección de discurso de odio...')
hate_results = df['texto_limpio'].progress_apply(run_hate)
df = pd.concat([df, pd.DataFrame(list(hate_results))], axis=1)
print('Listo.')
df[['texto_limpio', 'hate_label', 'hate_score']].head(5)

## 6. Visualización comparativa Boric vs Kast

In [ ]:
# Recargar resultados ya analizados (saltar secciones 4 y 5)
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

df = pd.read_csv('resultados_analisis_gobierno.csv', encoding='utf-8-sig')
df['fecha'] = pd.to_datetime(df['Fecha (dd/mm/yyyy)'], dayfirst=True, errors='coerce')
# Día relativo desde el inicio del mandato (11 marzo) — permite comparar 2022 vs 2026 en el mismo eje
df['mes_dia'] = df['fecha'].dt.strftime('%m-%d')
df['dia_mandato'] = (df['fecha'] - df.groupby('ID recolección')['fecha'].transform('min')).dt.days

GROUP_COL = 'ID recolección'
GROUPS = ['boric 1', 'kast 1']
GROUP_LABELS = {'boric 1': 'Boric (2022)', 'kast 1': 'Kast (2026)'}
GROUP_COLORS = {'Boric (2022)': '#1f77b4', 'Kast (2026)': '#d62728'}
df['Periodo'] = df[GROUP_COL].map(GROUP_LABELS)

print(f'{len(df)} filas cargadas.')
print(df['Periodo'].value_counts())

In [ ]:
# --- Gráfico 1: Distribución de sentimiento por período (barras apiladas %) ---
pivot_sent = (
    df.groupby(['Periodo', 'sent_label'])
    .size()
    .unstack(fill_value=0)
    .reindex(index=[GROUP_LABELS[g] for g in GROUPS])
    .reindex(columns=['POS', 'NEU', 'NEG'], fill_value=0)
)
pivot_pct = pivot_sent.div(pivot_sent.sum(axis=1), axis=0).mul(100).round(1).reset_index()
pivot_long = pivot_pct.melt(id_vars='Periodo', var_name='Sentimiento', value_name='Porcentaje')

label_map = {'POS': 'Positivo', 'NEU': 'Neutro', 'NEG': 'Negativo'}
pivot_long['Sentimiento'] = pivot_long['Sentimiento'].map(label_map)

fig1 = px.bar(
    pivot_long,
    x='Periodo', y='Porcentaje', color='Sentimiento',
    barmode='stack',
    color_discrete_map={'Positivo': '#4caf50', 'Neutro': '#9e9e9e', 'Negativo': '#f44336'},
    category_orders={'Sentimiento': ['Positivo', 'Neutro', 'Negativo']},
    text='Porcentaje',
    title='Distribución de sentimiento — Boric (2022) vs Kast (2026)',
)
fig1.update_traces(texttemplate='%{text:.0f}%', textposition='inside', textfont_size=12)
fig1.update_layout(
    xaxis_title='', yaxis_title='% de textos',
    yaxis_range=[0, 105],
    legend_title='Sentimiento',
    plot_bgcolor='white',
    height=450,
)
fig1.write_image('sentimiento_gobierno.png', scale=2)
fig1.show()

In [ ]:
# --- Gráfico 2: Heatmap de probabilidades promedio de sentimiento ---
heat_data = (
    df.groupby('Periodo')[['sent_pos', 'sent_neu', 'sent_neg']]
    .mean()
    .reindex([GROUP_LABELS[g] for g in GROUPS])
    .rename(columns={'sent_pos': 'Positivo', 'sent_neu': 'Neutro', 'sent_neg': 'Negativo'})
    .round(3)
)

fig2 = px.imshow(
    heat_data,
    text_auto='.2f',
    color_continuous_scale='RdYlGn',
    zmin=0, zmax=0.7,
    aspect='auto',
    title='Probabilidad promedio de sentimiento por período',
)
fig2.update_layout(
    xaxis_title='', yaxis_title='',
    coloraxis_colorbar_title='Prob.',
    height=320,
)
fig2.write_image('heatmap_sentimiento_gobierno.png', scale=2)
fig2.show()

In [ ]:
# --- Gráfico 3: Distribución del score de odio por período (boxplot) ---
fig3 = px.box(
    df,
    x='Periodo', y='hate_score',
    color='Periodo',
    category_orders={'Periodo': [GROUP_LABELS[g] for g in GROUPS]},
    color_discrete_map=GROUP_COLORS,
    points='outliers',
    title='Distribución del score de discurso de odio',
)
fig3.add_hline(
    y=0.5, line_dash='dash', line_color='red', opacity=0.5,
    annotation_text='umbral 0.5', annotation_position='top right',
)
fig3.update_layout(
    xaxis_title='', yaxis_title='Score de odio (0–1)',
    yaxis_range=[-0.05, 1.05],
    showlegend=False,
    plot_bgcolor='white',
    height=450,
)
fig3.write_image('odio_gobierno.png', scale=2)
fig3.show()

# Tabla resumen
resumen_odio = df.groupby('Periodo')['hate_score'].agg(
    Media='mean', Mediana='median',
    Desv_Est='std',
    Pct_odiosos=lambda x: (x >= 0.5).mean() * 100
).round(3).reindex([GROUP_LABELS[g] for g in GROUPS])
resumen_odio

In [ ]:
# --- Gráfico 4: Evolución diaria del sentimiento negativo (eje = día desde inauguración) ---
ts = (
    df.groupby(['Periodo', 'dia_mandato'])['sent_neg']
    .mean()
    .round(3)
    .reset_index()
)

fig4 = px.line(
    ts,
    x='dia_mandato', y='sent_neg', color='Periodo',
    markers=True,
    category_orders={'Periodo': [GROUP_LABELS[g] for g in GROUPS]},
    color_discrete_map=GROUP_COLORS,
    title='Evolución del sentimiento negativo por día de mandato',
    labels={'dia_mandato': 'Días desde el 11 de marzo', 'sent_neg': 'Prob. promedio NEG'},
)
fig4.update_layout(
    yaxis_range=[0, 1],
    plot_bgcolor='white',
    height=480,
    hovermode='x unified',
)
fig4.write_image('evolucion_sent_neg_gobierno.png', scale=2)
fig4.show()

In [ ]:
# --- Gráfico 5: Volumen de cobertura por día y período ---
vol = (
    df.groupby(['Periodo', 'dia_mandato'])
    .size().reset_index(name='n')
)

fig5 = px.line(
    vol,
    x='dia_mandato', y='n', color='Periodo',
    markers=True,
    color_discrete_map=GROUP_COLORS,
    title='Volumen diario de cobertura — Boric vs Kast',
    labels={'dia_mandato': 'Días desde el 11 de marzo', 'n': 'N° de notas'},
)
fig5.update_layout(plot_bgcolor='white', height=420, hovermode='x unified')
fig5.write_image('volumen_gobierno.png', scale=2)
fig5.show()

In [ ]:
# --- Gráfico 6: % de textos con discurso de odio por período ---
odio_pct = (
    df.groupby('Periodo')
    .apply(lambda g: (g['hate_label'] != 'not_hateful').mean() * 100, include_groups=False)
    .reindex([GROUP_LABELS[g] for g in GROUPS])
    .round(2)
    .reset_index()
)
odio_pct.columns = ['Periodo', 'pct_odio']

fig6 = px.bar(
    odio_pct,
    x='Periodo', y='pct_odio',
    color='Periodo',
    color_discrete_map=GROUP_COLORS,
    text='pct_odio',
    title='% de textos con discurso de odio',
    labels={'pct_odio': '% de textos'},
)
fig6.update_traces(texttemplate='%{text:.2f}%', textposition='outside')
fig6.update_layout(
    xaxis_title='', yaxis_title='% de textos',
    showlegend=False,
    plot_bgcolor='white',
    height=380,
)
fig6.write_image('pct_odio_gobierno.png', scale=2)
fig6.show()

In [ ]:
# --- Gráfico 7: Distribución por fuente y período ---
fuente_pivot = (
    df.groupby(['Fuente', 'Periodo']).size()
    .unstack(fill_value=0)
    .reindex(columns=[GROUP_LABELS[g] for g in GROUPS], fill_value=0)
    .reset_index()
)
fuente_long = fuente_pivot.melt(id_vars='Fuente', var_name='Periodo', value_name='n')

fig7 = px.bar(
    fuente_long,
    x='Fuente', y='n', color='Periodo',
    barmode='group',
    color_discrete_map=GROUP_COLORS,
    text='n',
    title='Cobertura por fuente — Boric vs Kast',
)
fig7.update_traces(textposition='outside')
fig7.update_layout(plot_bgcolor='white', height=420, yaxis_title='N° de notas')
fig7.write_image('fuentes_gobierno.png', scale=2)
fig7.show()

## 7. Guardado de resultados

In [ ]:
output_cols = [
    'Tipo', 'Fecha (dd/mm/yyyy)', 'Fuente', 'Búsqueda Original',
    'ID recolección', 'Orientación política', 'texto_limpio',
    'sent_label', 'sent_pos', 'sent_neu', 'sent_neg',
    'hate_label', 'hate_score'
]
output_cols = [c for c in output_cols if c in df.columns]

df[output_cols].to_csv('resultados_analisis_gobierno.csv', index=False, encoding='utf-8-sig')
print(f'Guardado: resultados_analisis_gobierno.csv  ({len(df)} filas)')

print('\n--- Resumen de sentimiento (global) ---')
print(df['sent_label'].value_counts(normalize=True).map('{:.1%}'.format))

print('\n--- Resumen de sentimiento por período ---')
print(df.groupby('ID recolección')['sent_label'].value_counts(normalize=True).map('{:.1%}'.format))

print('\n--- Resumen de discurso de odio (global) ---')
print(df['hate_label'].value_counts(normalize=True).map('{:.1%}'.format))